In [ ]:
import zipfile
import os

zip_path = "/content/archive (2).zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

In [ ]:
os.listdir("/content/dataset")

In [ ]:
os.listdir("/content/dataset/train")

In [ ]:
import os

base_path = "/content/dataset/train"

for class_name in os.listdir(base_path):
    class_path = os.path.join(base_path, class_name)

    if os.path.isdir(class_path):
        count = len(os.listdir(class_path))
        print(f"{class_name}: {count} images")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
import random

base_path = "/content/dataset/train"

classes = ["yawn", "Closed", "Open", "no_yawn"]

plt.figure(figsize=(12, 8))

for i, class_name in enumerate(classes):
    class_path = os.path.join(base_path, class_name)

    image_name = random.choice(os.listdir(class_path))
    image_path = os.path.join(class_path, image_name)

    image = Image.open(image_path)

    plt.subplot(2, 2, i + 1)
    plt.imshow(image)
    plt.title(class_name)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import shutil
import random

source_dir = "/content/dataset/train"
output_dir = "/content/drowsiness_dataset"

# Create train, validation and test folders
for split in ["train", "validation", "test"]:
    for class_name in ["yawn", "Closed", "Open", "no_yawn"]:
        os.makedirs(
            os.path.join(output_dir, split, class_name),
            exist_ok=True
        )

# Set random seed so the split is reproducible
random.seed(42)

# Split each class
for class_name in ["yawn", "Closed", "Open", "no_yawn"]:

    class_path = os.path.join(source_dir, class_name)

    images = [
        file for file in os.listdir(class_path)
        if os.path.isfile(os.path.join(class_path, file))
    ]

    random.shuffle(images)

    total = len(images)

    train_end = int(total * 0.80)
    validation_end = train_end + int(total * 0.10)

    train_images = images[:train_end]
    validation_images = images[train_end:validation_end]
    test_images = images[validation_end:]

    # Copy training images
    for image in train_images:
        shutil.copy(
            os.path.join(class_path, image),
            os.path.join(output_dir, "train", class_name, image)
        )

    # Copy validation images
    for image in validation_images:
        shutil.copy(
            os.path.join(class_path, image),
            os.path.join(output_dir, "validation", class_name, image)
        )

    # Copy testing images
    for image in test_images:
        shutil.copy(
            os.path.join(class_path, image),
            os.path.join(output_dir, "test", class_name, image)
        )

print("Dataset split completed successfully!")

In [ ]:
for split in ["train", "validation", "test"]:
    print("\n", split.upper())

    for class_name in ["yawn", "Closed", "Open", "no_yawn"]:
        path = os.path.join(output_dir, split, class_name)
        count = len(os.listdir(path))
        print(f"{class_name}: {count}")

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Image size
IMG_SIZE = (224, 224)

# Batch size
BATCH_SIZE = 32

# Dataset paths
train_dir = "/content/drowsiness_dataset/train"
validation_dir = "/content/drowsiness_dataset/validation"
test_dir = "/content/drowsiness_dataset/test"

print("TensorFlow version:", tf.__version__)

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True
)

validation_datagen = ImageDataGenerator(
    rescale=1./255
)

test_datagen = ImageDataGenerator(
    rescale=1./255
)

print("Data preprocessing configuration completed!")

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=42
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

In [ ]:
print("Class mapping:")
print(train_generator.class_indices)

In [ ]:
import matplotlib.pyplot as plt

images, labels = next(train_generator)

plt.figure(figsize=(10, 8))

for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout

custom_cnn = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(224, 224, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.5),

    Dense(4, activation="softmax")
])

custom_cnn.summary()

In [ ]:
custom_cnn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Custom CNN compiled successfully!")

In [ ]:
history_cnn = custom_cnn.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)

In [ ]:
import matplotlib.pyplot as plt

# Accuracy graph
plt.figure(figsize=(8, 5))

plt.plot(history_cnn.history["accuracy"], label="Training Accuracy")
plt.plot(history_cnn.history["val_accuracy"], label="Validation Accuracy")

plt.title("Custom CNN - Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.show()


# Loss graph
plt.figure(figsize=(8, 5))

plt.plot(history_cnn.history["loss"], label="Training Loss")
plt.plot(history_cnn.history["val_loss"], label="Validation Loss")

plt.title("Custom CNN - Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
test_loss, test_accuracy = custom_cnn.evaluate(test_generator)

print("Custom CNN Test Loss:", test_loss)
print("Custom CNN Test Accuracy:", test_accuracy)
print("Test Accuracy (%):", test_accuracy * 100)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Reset test generator
test_generator.reset()

# Predictions
predictions = custom_cnn.predict(test_generator)

# Convert probabilities to class numbers
predicted_classes = np.argmax(predictions, axis=1)

# Actual class numbers
true_classes = test_generator.classes

# Class names
class_names = list(test_generator.class_indices.keys())

# Confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)

# Display
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot()
plt.title("Custom CNN - Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    true_classes,
    predicted_classes,
    target_names=class_names
)

print("Custom CNN Classification Report")
print(report)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

print("MobileNetV2 base model loaded successfully!")
print("Trainable:", base_model.trainable)

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Dense, Dropout

x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)

output = Dense(4, activation="softmax")(x)

mobilenet_model = Model(
    inputs=base_model.input,
    outputs=output
)

mobilenet_model.summary()

In [ ]:
mobilenet_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("MobileNetV2 model compiled successfully!")

In [ ]:
history_mobilenet = mobilenet_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)

In [ ]:
import matplotlib.pyplot as plt

# Accuracy Graph
plt.figure(figsize=(8, 5))

plt.plot(
    history_mobilenet.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history_mobilenet.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("MobileNetV2 - Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()


# Loss Graph
plt.figure(figsize=(8, 5))

plt.plot(
    history_mobilenet.history["loss"],
    label="Training Loss"
)

plt.plot(
    history_mobilenet.history["val_loss"],
    label="Validation Loss"
)

plt.title("MobileNetV2 - Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
mobilenet_test_loss, mobilenet_test_accuracy = mobilenet_model.evaluate(
    test_generator
)

print("MobileNetV2 Test Loss:", mobilenet_test_loss)
print("MobileNetV2 Test Accuracy:", mobilenet_test_accuracy)
print("Test Accuracy (%):", mobilenet_test_accuracy * 100)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Reset test generator
test_generator.reset()

# Predictions
mobilenet_predictions = mobilenet_model.predict(test_generator)

# Convert probabilities to class numbers
mobilenet_predicted_classes = np.argmax(
    mobilenet_predictions,
    axis=1
)

# Actual classes
true_classes = test_generator.classes

# Class names
class_names = list(test_generator.class_indices.keys())

# Create confusion matrix
cm_mobilenet = confusion_matrix(
    true_classes,
    mobilenet_predicted_classes
)

# Display confusion matrix
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_mobilenet,
    display_labels=class_names
)

disp.plot()
plt.title("MobileNetV2 - Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

mobilenet_report = classification_report(
    true_classes,
    mobilenet_predicted_classes,
    target_names=class_names
)

print("MobileNetV2 Classification Report")
print(mobilenet_report)

In [ ]:
print("===== MODEL COMPARISON =====")

print("Custom CNN Test Accuracy   :", round(test_accuracy * 100, 2), "%")
print("MobileNetV2 Test Accuracy  :", round(mobilenet_test_accuracy * 100, 2), "%")

improvement = (mobilenet_test_accuracy - test_accuracy) * 100

print("Accuracy Improvement       :", round(improvement, 2), "percentage points")

In [ ]:
def fatigue_level(predicted_class):

    if predicted_class in ["Open", "no_yawn"]:
        return 0, "Alert"

    elif predicted_class == "yawn":
        return 1, "Mild Fatigue"

    elif predicted_class == "Closed":
        return 2, "Severe Fatigue"


# Test the decision logic
test_classes = ["Open", "no_yawn", "yawn", "Closed"]

for cls in test_classes:
    level, status = fatigue_level(cls)
    print(cls, "→", level, "→", status)

In [ ]:
import matplotlib.pyplot as plt

# Simulated sequential predictions from a driving session
predicted_sequence = [
    "Open",
    "no_yawn",
    "Open",
    "no_yawn",
    "yawn",
    "yawn",
    "no_yawn",
    "yawn",
    "Closed",
    "Closed"
]

# Convert predictions into fatigue levels
fatigue_levels = []

for prediction in predicted_sequence:
    level, status = fatigue_level(prediction)
    fatigue_levels.append(level)

# Create time/frame sequence
frames = list(range(1, len(fatigue_levels) + 1))

# Plot fatigue progression
plt.figure(figsize=(10, 5))

plt.plot(
    frames,
    fatigue_levels,
    marker="o"
)

plt.yticks(
    [0, 1, 2],
    ["Alert", "Mild Fatigue", "Severe Fatigue"]
)

plt.xlabel("Frame / Time Sequence")
plt.ylabel("Fatigue Level")
plt.title("Driver Fatigue Progression Curve")

plt.grid(True)
plt.show()

In [ ]:
print("===== PERFORMANCE ANALYSIS =====")

print("Custom CNN Accuracy     :", round(test_accuracy * 100, 2), "%")
print("MobileNetV2 Accuracy    :", round(mobilenet_test_accuracy * 100, 2), "%")

print("\nMobileNetV2 Class Performance:")
print("Closed   - Precision: 0.99 | Recall: 0.99 | F1: 0.99")
print("Open     - Precision: 0.99 | Recall: 0.99 | F1: 0.99")
print("no_yawn  - Precision: 0.71 | Recall: 0.99 | F1: 0.83")
print("yawn     - Precision: 0.98 | Recall: 0.60 | F1: 0.75")

print("\nFatigue Decision Levels:")
print("0 → Alert")
print("1 → Mild Fatigue")
print("2 → Severe Fatigue")

print("\nPerformance analysis completed successfully!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

# Get one batch from test data
test_generator.reset()
images, labels = next(test_generator)

# Predict using MobileNetV2
predictions = mobilenet_model.predict(images)

predicted_indices = np.argmax(predictions, axis=1)
actual_indices = np.argmax(labels, axis=1)

class_names = list(test_generator.class_indices.keys())

# Display 8 sample predictions
plt.figure(figsize=(12, 8))

for i in range(8):
    predicted_class = class_names[predicted_indices[i]]
    actual_class = class_names[actual_indices[i]]

    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i])
    plt.title(
        f"Actual: {actual_class}\nPredicted: {predicted_class}"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Save Custom CNN
custom_cnn.save("/content/custom_cnn_model.keras")

# Save MobileNetV2
mobilenet_model.save("/content/mobilenetv2_model.keras")

print("Both models saved successfully!")

In [ ]:
import os

print("Files in /content:")

for file in os.listdir("/content"):
    print(file)

In [ ]:
import os

project_dir = "/content/Driver_Drowsiness_Detection"

folders = [
    "notebooks",
    "models",
    "results",
    "documentation"
]

for folder in folders:
    os.makedirs(os.path.join(project_dir, folder), exist_ok=True)

print("Project folder created successfully!")

for root, dirs, files in os.walk(project_dir):
    print(root)

In [ ]:
import shutil

project_dir = "/content/Driver_Drowsiness_Detection"

# Copy Custom CNN model
shutil.copy(
    "/content/custom_cnn_model.keras",
    project_dir + "/models/custom_cnn_model.keras"
)

# Copy MobileNetV2 model
shutil.copy(
    "/content/mobilenetv2_model.keras",
    project_dir + "/models/mobilenetv2_model.keras"
)

print("Both trained models copied successfully!")

In [ ]:
import os

models_path = "/content/Driver_Drowsiness_Detection/models"

print("Files inside models folder:")

for file in os.listdir(models_path):
    print(file)

In [ ]:
import matplotlib.pyplot as plt
import os

results_dir = "/content/Driver_Drowsiness_Detection/results"

plt.figure(figsize=(12, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history_mobilenet.history["accuracy"], label="Training Accuracy")
plt.plot(history_mobilenet.history["val_accuracy"], label="Validation Accuracy")
plt.title("MobileNetV2 Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history_mobilenet.history["loss"], label="Training Loss")
plt.plot(history_mobilenet.history["val_loss"], label="Validation Loss")
plt.title("MobileNetV2 Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(results_dir, "mobilenetv2_training_history.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("MobileNetV2 training graph saved successfully!")

In [ ]:
print("Available training history variables:")

for name in ["history", "history_custom", "history_mobilenet"]:
    print(name, "→", name in globals())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Test data reset
test_generator.reset()

# Predictions
predictions = mobilenet_model.predict(test_generator)

predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes

# Class names
class_names = list(test_generator.class_indices.keys())

# Confusion Matrix
cm = confusion_matrix(true_classes, predicted_classes)

# Display
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax)
plt.title("MobileNetV2 Confusion Matrix")
plt.tight_layout()

# Save
plt.savefig(
    "/content/Driver_Drowsiness_Detection/results/mobilenetv2_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("MobileNetV2 confusion matrix saved successfully!")

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

# Classification report
report = classification_report(
    true_classes,
    predicted_classes,
    target_names=class_names,
    output_dict=True
)

# Convert to DataFrame
report_df = pd.DataFrame(report).transpose()

# Save as CSV
report_df.to_csv(
    "/content/Driver_Drowsiness_Detection/results/mobilenetv2_classification_report.csv"
)

print("Classification report saved successfully!")

In [ ]:
results_summary = """
DRIVER DROWSINESS DETECTION
================================

Model Performance
-----------------
Custom CNN Test Accuracy   : 82.65%
MobileNetV2 Test Accuracy  : 89.12%
Accuracy Improvement       : 6.46 percentage points

MobileNetV2 Class Performance
-----------------------------
Closed   - Precision: 0.99 | Recall: 0.99 | F1: 0.99
Open     - Precision: 0.99 | Recall: 0.99 | F1: 0.99
no_yawn  - Precision: 0.71 | Recall: 0.99 | F1: 0.83
yawn     - Precision: 0.98 | Recall: 0.60 | F1: 0.75

Fatigue Decision Levels
-----------------------
0 -> Alert
1 -> Mild Fatigue
2 -> Severe Fatigue

Dataset
-------
Total Images : 2900
Training     : 2318
Validation   : 288
Testing      : 294
Classes      : Closed, Open, no_yawn, yawn
"""

with open(
    "/content/Driver_Drowsiness_Detection/results/results_summary.txt",
    "w"
) as file:
    file.write(results_summary)

print("Results summary saved successfully!")